In [30]:
### Connect to getsongbpm
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("/Users/bytedance/Documents/Mauriciio/Jupyter/Metallica/.env")

GETSONGBPM_API_KEY = os.getenv("GETSONGBPM_API_KEY")

big4 = pd.read_csv("big4_tracks_enriched_lastfm.csv")

In [39]:
def get_bpm_from_getsongbpm(artist, track):
    url = "https://api.getsong.co/search/"
    
    params = {
        "api_key": GETSONGBPM_API_KEY,
        "type": "both",
        "lookup": f"song:{track} artist:{artist}",
        "limit": 5
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        return {
            "bpm": None,
            "key_of": None,
            "getsongbpm_match_title": None,
            "getsongbpm_match_artist": None,
            "getsongbpm_status": response.status_code
        }

    data = response.json()
    results = data.get("search", [])

    if isinstance(results, dict):
        results = list(results.values())

    if not isinstance(results, list) or len(results) == 0:
        return {
            "bpm": None,
            "key_of": None,
            "getsongbpm_match_title": None,
            "getsongbpm_match_artist": None,
            "getsongbpm_status": 200
        }

    best = results[0]

    if not isinstance(best, dict):
        return {
            "bpm": None,
            "key_of": None,
            "getsongbpm_match_title": None,
            "getsongbpm_match_artist": None,
            "getsongbpm_status": 200
        }

    artist_info = best.get("artist", {})
    if not isinstance(artist_info, dict):
        artist_info = {}

    return {
        "bpm": best.get("tempo"),
        "key_of": best.get("key_of"),
        "getsongbpm_match_title": best.get("title"),
        "getsongbpm_match_artist": artist_info.get("name"),
        "getsongbpm_status": 200
    }

In [40]:
bpm_rows = []

for i, row in big4.iterrows():
    artist = row["artist"]
    track = row["track_name"]
    
    print(f"{i+1}/{len(big4)} - {artist} - {track}")
    
    try:
        bpm_info = get_bpm_from_getsongbpm(artist, track)
    except Exception as e:
        print("ERROR:", artist, "-", track, "|", e)
        bpm_info = {
            "bpm": None,
            "key_of": None,
            "getsongbpm_match_title": None,
            "getsongbpm_match_artist": None,
            "getsongbpm_status": "error"
        }
    
    bpm_rows.append({
        "artist": artist,
        "track_name": track,
        **bpm_info
    })
    
    time.sleep(0.4)

1/813 - Metallica - Hit the Lights
2/813 - Metallica - Hit the Lights
3/813 - Metallica - The Four Horsemen
4/813 - Metallica - The Four Horsemen
5/813 - Metallica - Motorbreath
6/813 - Metallica - Jump in the Fire
7/813 - Metallica - Jump in the Fire
8/813 - Metallica - (Anesthesia) Pulling Teeth
9/813 - Metallica - Whiplash
10/813 - Metallica - Phantom Lord
11/813 - Metallica - No Remorse
12/813 - Metallica - Seek & Destroy
13/813 - Metallica - Metal Militia
14/813 - Metallica - Metal Militia
15/813 - Metallica - Fight Fire With Fire
16/813 - Metallica - Ride the Lightning
17/813 - Metallica - Ride the Lightning
18/813 - Metallica - For Whom the Bell Tolls
19/813 - Metallica - For Whom the Bell Tolls
20/813 - Metallica - Fade to Black
21/813 - Metallica - Fade to Black
22/813 - Metallica - Trapped Under Ice
23/813 - Metallica - Escape
24/813 - Metallica - Creeping Death
25/813 - Metallica - Creeping Death
26/813 - Metallica - The Call of Ktulu
27/813 - Metallica - Battery
28/813 - Me

In [43]:
big4_bpm.duplicated(subset=["artist", "track_name"]).sum()

np.int64(1390)

In [44]:
dups = big4_bpm[big4_bpm.duplicated(subset=["artist", "track_name"], keep=False)]

dups.sort_values(["artist", "track_name"]).head(20)

,artist,album_name,first_release_date,release_group_id,release_id,recording_id,track_number,track_position,track_name,track_length_ms,lastfm_listeners,lastfm_playcount,lastfm_url,bpm,key_of,getsongbpm_match_title,getsongbpm_match_artist,getsongbpm_status
1505,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1506,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1507,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1508,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1509,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1510,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1511,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1512,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1513,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200
1514,Anthrax,Spreading the Disease,1985-10-30,5ebc6b4f-6b03-3202-86ac-a999408143fe,23bbb622-de3e-4e32-b059-937ff4f042fe,4700ab16-23c9-411d-9514-6ad750595604,A1,1,A.I.R.,345333.0,153019,740059,https://www.last.fm/music/Anthrax/_/A.I.R.,99,B,A.I.R.,Anthrax,200


In [45]:
dups[["artist", "track_name", "album_name"]].sort_values(["artist", "track_name"])

,artist,track_name,album_name
1505,Anthrax,A.I.R.,Spreading the Disease
1506,Anthrax,A.I.R.,Spreading the Disease
1507,Anthrax,A.I.R.,Spreading the Disease
1508,Anthrax,A.I.R.,Spreading the Disease
1509,Anthrax,A.I.R.,Spreading the Disease
...,...,...,...
1095,Slayer,War Ensemble,Repentless
1096,Slayer,War Ensemble,Repentless
1097,Slayer,War Ensemble,Repentless
1098,Slayer,War Ensemble,Repentless


In [46]:
# Check duplicates within the same artist + album + track
album_track_dups = big4_bpm[
    big4_bpm.duplicated(
        subset=["artist", "album_name", "track_name"],
        keep=False
    )
]

album_track_dups[[
    "artist", "album_name", "track_name", "bpm", 
    "getsongbpm_match_title", "getsongbpm_match_artist"
]].sort_values(["artist", "album_name", "track_name"])

,artist,album_name,track_name,bpm,getsongbpm_match_title,getsongbpm_match_artist
1566,Anthrax,Among the Living,Among the Living,156,Among the Living,Anthrax
1567,Anthrax,Among the Living,Among the Living,156,Among the Living,Anthrax
1568,Anthrax,Among the Living,Among the Living,156,Among the Living,Anthrax
1569,Anthrax,Among the Living,Among the Living,156,Among the Living,Anthrax
1570,Anthrax,Among the Living,Among the Living,156,Among the Living,Anthrax
...,...,...,...,...,...,...
1016,Slayer,World Painted Blood,Hate Worldwide,105,Hate Worldwide,Slayer
1017,Slayer,World Painted Blood,Hate Worldwide,105,Hate Worldwide,Slayer
1018,Slayer,World Painted Blood,Hate Worldwide,105,Hate Worldwide,Slayer
1019,Slayer,World Painted Blood,Hate Worldwide,105,Hate Worldwide,Slayer


In [47]:
big4_bpm_clean = big4_bpm.drop_duplicates(
    subset=["artist", "album_name", "track_name"],
    keep="first"
).copy()

In [48]:
print("Before:", big4_bpm.shape)
print("After:", big4_bpm_clean.shape)

big4_bpm_clean.duplicated(
    subset=["artist", "album_name", "track_name"]
).sum()

Before: (1989, 18)
After: (652, 18)


np.int64(0)

In [49]:
big4_bpm_clean.to_csv("big4_tracks_with_bpm_clean.csv", index=False)

In [50]:
## Coverafge
coverage = big4_bpm_clean["bpm"].notna().mean()
print(f"BPM coverage: {coverage:.2%}")

BPM coverage: 82.06%
